# Implementation of the DDPM (Denoising Diffusion Probabilistic Models) in PyTorch
This implementation is based on the original paper "Denoising Diffusion Probabilistic Models" by Ho et al. (2020).

## Architectural Overview
The DDPM architecture consists of two main components: the forward diffusion process and the reverse denoising process.

### 1. Forward Diffusion Process
This process gradually adds noise to the data over a series of time steps. The noise is added according to a predefined schedule, which can be linear or cosine-based. The forward process can be mathematically represented as:

$$
q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t I)
$$

where $\beta_t$ is the noise variance at time step $t$.

### 2. Reverse Denoising Process
This process aims to reverse the noise added in the forward process. It is parameterized by a neural network (often a U-Net architecture) that predicts the noise added at each time step. The reverse process can be represented as:

$$
p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))
$$

where $\mu_\theta$ and $\Sigma_\theta$ are the mean and covariance predicted by the neural network.

## Mathematical Details

### Key Properties:
- The diffusion process is a continuous-time Stochastic Markov process.
- The noise schedule in the original paper is linear (from 0.0001 to 0.02) or cosine-based.

### Important Notes on Reverse Diffusion:

1. **Intractability of the Exact Reverse Process:**
   
   The exact reverse process is a mixture of Gaussians and is intractable, represented as:
   
   $$
   p(x_{t-1} | x_t) = \int p(x_{t-1} | x_t, x_0) q(x_0 | x_t) dx_0
   $$

2. **Gaussian Approximation:**
   
   In DDPM, we approximate the reverse process by a single Gaussian distribution parameterized by a neural network:
   
   $$
   p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))
   $$
   
   The mean and covariance are predicted based on the noisy input $x_t$ and time step $t$.

3. **Fixed Covariance:**
   
   The model covariance is fixed to match the forward process noise schedule, simplifying training. The neural network only predicts the mean $\mu_\theta(x_t, t)$.

### Training Objective:

The ELBO (Evidence Lower Bound) can be derived as:

$$
\mathcal{L}_{\text{VLB}} = \mathbb{E}_{q} \left[ D_{\text{KL}}(q(x_T | x_0) || p(x_T)) + \sum_{t=2}^T D_{\text{KL}}(q(x_{t-1} | x_t, x_0) || p_\theta(x_{t-1} | x_t)) - \log p_\theta(x_0 | x_1) \right]
$$

This can be simplified. Using the reparameterization trick, we can express:

$$
x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

where $\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$.

The true reverse posterior is:

$$
q(x_{t-1} | x_t, x_0) = \mathcal{N}(x_{t-1}; \tilde{\mu}_t(x_t, x_0), \tilde{\beta}_t I)
$$

where:

$$
\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}} \beta_t}{1 - \bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t
$$

$$
\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t
$$

### Simplified Loss Function:

The paper shows that minimizing the KL divergence is equivalent to predicting the noise $\epsilon$. The simplified training objective becomes:

$$
\mathcal{L}_{\text{simple}} = \mathbb{E}_{t \sim [1,T], x_0, \epsilon \sim \mathcal{N}(0,I)} \left[ ||\epsilon - \epsilon_\theta(x_t, t)||^2 \right]
$$

where:
- $\epsilon$ is the actual noise added to the data at time step $t$
- $\epsilon_\theta(x_t, t)$ is the noise predicted by the neural network
- $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$

This simplified loss removes the weighting factors for better empirical performance.

In [1]:
-ls /kaggle/working

NameError: name 'ls' is not defined

In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
from torchvision import transforms
from PIL import Image

In [ ]:
from torch.utils.data import Dataset
from torchvision.io import read_image
from glob import glob
import os

class CelebADatasetHQ(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.image_paths = sorted(glob(os.path.join(root_dir, "*.jpg")))
        if not self.image_paths:
            raise ValueError("No images found.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        img = Image.open(image_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

In [ ]:
# mean = torch.tensor([0.5061, 0.4254, 0.3828])
# std  = torch.tensor([0.2659, 0.2452, 0.2413])
#TODO : normalization to [-1,1]
transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(128),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

inv_normalize = transforms.Normalize(
    (-1.0, -1.0, -1.0),
    (2.0, 2.0, 2.0)
)

img shape : 178x218

In [ ]:
dataset = CelebADatasetHQ(root_dir=r"/kaggle/input/datasets/woodstoc123k/celeba-64k/celeba/img_align_celeba", transform=transform)
 # Check the length of the dataset

In [ ]:
print(f"dataset length: {len(dataset)}")
dataset[0].min(), dataset[0].max()

In [ ]:
class TimeEmbedding(nn.Module):
    """
    Sinusoidal time embedding for diffusion timesteps with MLP projection.
    Maps timestep t to a high-dimensional embedding using sinusoidal functions + MLP.
    """
    def __init__(self, dim):
        super(TimeEmbedding, self).__init__()
        assert dim % 2 == 0, "Dimension must be even"
        self.dim = dim
        
        # MLP projection after sinusoidal embedding
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )
        
    def forward(self, t):
        """
        Args:
            t: Tensor of shape (B,) containing timesteps
        Returns:
            Time embedding of shape (B, dim)
        """
        device = t.device
        half_dim = self.dim // 2
        
        # Create sinusoidal embeddings: 1 / (10000^(2i/dim))
        freqs = torch.exp(
            -torch.log(torch.tensor(10000.0)) * torch.arange(0, half_dim, dtype=torch.float32, device=device) / half_dim
        )

        args = t.unsqueeze(1) * freqs.unsqueeze(0)

        sinusoidal_emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        time_emb = self.mlp(sinusoidal_emb)
        return time_emb

In [ ]:
class DownSample(nn.Module):
    """
    Downsample spatial dimensions by factor of 2 using strided convolution.
    (B, C, H, W) -> (B, C_out, H/2, W/2)
    """
    def __init__(self, in_channels, out_channels):
        super(DownSample, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
        
    def forward(self, x):
        return self.conv(x)


class UpSample(nn.Module):
    """
    Upsample spatial dimensions by factor of 2 using nearest-neighbor interpolation + convolution.
    (B, C, H, W) -> (B, C_out, H*2, W*2)
    """
    def __init__(self, in_channels, out_channels):
        super(UpSample, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        
    def forward(self, x):
        x = torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)

In [ ]:
class ResBlockFiLM(nn.Module):
    """
    Residual block with FiLM (Feature-wise Linear Modulation) conditioning (STANDARD SINGLE FiLM).
    
    Architecture:
        x -> Conv1 -> GroupNorm1 -> SiLU -> Conv2 -> GroupNorm2 -> FiLM -> SiLU -> + residual
    
    FiLM applies time-aware modulation: y = γ ⊙ x + β
    where γ,β are generated from time embedding
    """
    def __init__(self, in_channels, out_channels, time_emb_dim=256, groups=8, dropout=0.0):
        super(ResBlockFiLM, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        assert self.out_channels % self.groups == 0, "out_channels must be divisible by groups"
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=1,padding=1)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
        # GroupNorm layers (after each conv)
        self.groupnorm1 = nn.GroupNorm(groups,out_channels)
        self.groupnorm2 = nn.GroupNorm(groups,out_channels)
        
        # dROPOUT REGULARIZATION
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        # FiLM parameter generator (SINGLE PAIR: gamma, beta for modulation after conv2)
        self.film = nn.Linear(time_emb_dim, out_channels * 2)
        
        # Residual connection 
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=1)
        else:
            self.residual_conv = nn.Identity()
    def forward(self,x,emb):
        """ 
        x: feature map of shape (B,C,H,W)
        emb: time embedding of shape (B,time_emb_dim)
        """
        # Residual path
        residual = self.residual_conv(x)
        
        # Generate FiLM parameters from time embedding
        conditioner = self.film(emb)  # (B, out_channels * 2)
        gamma, beta = conditioner.chunk(2, dim=-1)  # Each: (B, out_channels)
        
        # Reshape for broadcasting: (B, C) -> (B, C, 1, 1)
        gamma = gamma[:, :, None, None]
        beta = beta[:, :, None, None]
        
        h = self.conv1(x)
        h = self.groupnorm1(h)
        h = F.silu(h)
        h = self.dropout(h)
        
        h = self.conv2(h)
        h = self.groupnorm2(h)
        h = gamma * h + beta  # FiLM modulation 
        h = F.silu(h)
        
        # Residual connection (skip connection)
        return h + residual


In [ ]:
class SelfAttention(nn.Module):
    """ 
    Self-Attention block for capturing long-range dependencies in feature maps.
    Architecture:
        x -> GroupNorm -> QKV projection -> Scaled Dot-Product Attention -> Output projection -> + residual
    """
    def __init__(self, in_channels, num_heads, groups=8):
        super(SelfAttention, self).__init__()
        assert in_channels % num_heads == 0, "numbers of channels must be divisible by number of attention heads!.."
        assert in_channels % groups == 0, "in_channels must be divisible by groups"
        self.in_channels = in_channels
        self.n_heads = num_heads
        self.head_dim = self.in_channels // num_heads
        self.groups = groups
        self.norm = nn.GroupNorm(groups, in_channels)

        # for self attention we'll use scaled dot-product attention
        self.qkv_proj = nn.Linear(in_channels, 3 * in_channels)
        self.out_proj = nn.Linear(in_channels, in_channels)

    def forward(self, x):
        if x.dim() != 4:
            raise ValueError("SelfAttention expects input of shape (B, C, H, W)")

        B, C, H, W = x.shape
        residual = x

        x = self.norm(x)
        x = x.flatten(2).transpose(1, 2)  # (B, T, C) where T = H*W

        qkv = self.qkv_proj(x)  # (B, T, 3C)
        qkv = qkv.view(B, -1, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, n_heads, T, head_dim)
        q, k, v = qkv.unbind(0)  # Each: (B, n_heads, T, head_dim)

        out = F.scaled_dot_product_attention(q, k, v)  # (B, n_heads, T, head_dim)
        out = out.permute(0, 2, 1, 3).contiguous().view(B, -1, C)  # (B, T, C)
        out = self.out_proj(out)

        out = out.transpose(1, 2).reshape(B, C, H, W)
        return residual + out


In [ ]:
class EncoderBlock(nn.Module):
    """ 
    Encoder block for U-Net downsampling path.
    
    Structure:
        - Multiple Residual Blocks with FiLM conditioning
        - Optional self-attention blocks INTERLEAVED with residual blocks (1 per residual block)
        - Downsampling at the end
    
    Flow: ResBlock → [Attn] → ResBlock → [Attn] → ... → Downsample
    
    Why interleaved?
        - Standard in Stable Diffusion & modern diffusion models
        - Better information flow between residual and attention layers
        - More efficient: 1 attention per residual, not multiple in sequence
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, num_res_blocks=2, 
                 downsample=True, use_attention=False, num_heads=8, groups=8, dropout=0.0):
        super(EncoderBlock, self).__init__()
        
        self.use_attention = use_attention
        self.downsample = downsample
        
        # Create residual blocks
        self.res_blocks = nn.ModuleList([
            ResBlockFiLM(
                in_channels = in_channels if i == 0 else out_channels,
                out_channels = out_channels,
                time_emb_dim = time_emb_dim,
                groups = groups,
                dropout = dropout
        ) for i in range(num_res_blocks) 
        ])
        
        # Create attention blocks
        self.attention_blocks = nn.ModuleList()
        if use_attention:
            for _ in range(num_res_blocks):
                self.attention_blocks.append(SelfAttention(out_channels, num_heads, groups=groups))
        
        # Downsampling layer
        self.downsample_layer = DownSample(out_channels, out_channels) if downsample else nn.Identity()
    
    def forward(self, x, time_emb):
        """
        Forward pass with INTERLEAVED attention.
        
        Args:
            x: Feature map (B, in_channels, H, W)
            time_emb: Time embedding (B, time_emb_dim)
            
        Returns:
            x_down: Downsampled features (B, out_channels, H/2, W/2)
            x_skip: Features before downsampling (for skip connections)
        """
        # Process residual blocks with interleaved attention
        for i, res_block in enumerate(self.res_blocks):
            x = res_block(x, time_emb)
            if self.use_attention:
                x = self.attention_blocks[i](x)
    
        x_skip = x
        
        # Downsample
        x_down = self.downsample_layer(x)
        
        return x_down, x_skip  # Return both downsampled and pre-downsampled features for skip connections


In [ ]:
class DecoderBlock(nn.Module):
    """
    Decoder block for U-Net upsampling path.
    
    Structure:
        - Upsample input
        - Concatenate with skip connection from encoder
        - Multiple residual blocks with time conditioning
        - Optional self-attention blocks INTERLEAVED with residuals
    
    Flow: Upsample → Concat → ResBlock → [Attn] → ResBlock → [Attn] → Output

    """
    def __init__(self, in_channels, out_channels, time_emb_dim, num_res_blocks=2,
                 upsample=True, use_attention=False, num_heads=4, groups=8, dropout=0.0):
        super(DecoderBlock, self).__init__()
        
        # Upsampling
        self.upsample = UpSample(in_channels, out_channels) if upsample else nn.Identity()
        
        # Residual blocks (first block takes concatenated input of 2x channels)
        self.res_blocks = nn.ModuleList([
            ResBlockFiLM(
                in_channels = out_channels * 2 if i == 0 else out_channels,  # *2 for skip concatenation
                out_channels =  out_channels,
                time_emb_dim = time_emb_dim,
                groups = groups,
                dropout = dropout                
            ) for i in range(num_res_blocks)
        ])
        
        # Create attention blocks 
        self.use_attention = use_attention
        self.attention_blocks = nn.ModuleList()
        if self.use_attention:
            for _ in range(num_res_blocks):
                self.attention_blocks.append(SelfAttention(out_channels, num_heads, groups=groups))
    
    def forward(self, x, skip, time_emb):
        """
        Forward pass with optional attention.
        
        Args:
            x: Input from previous decoder block (B, in_channels, H, W)
            skip: Skip connection from encoder (B, out_channels, H*2, W*2)
            time_emb: Time embedding (B, time_emb_dim)
            
        Returns:
            Output tensor (B, out_channels, H*2, W*2)
        """
        # Upsample and concatenate with skip
        x = self.upsample(x)
        
        # Concatenate with skip connection from encoder
        x = torch.cat([x, skip], dim=1)  # (B, out_channels*2, H*2, W*2)
        
        # Apply residual blocks and optional attention
        for i, res_block in enumerate(self.res_blocks):
            x = res_block(x, time_emb)
            # Apply attention after each residual block (if enabled)
            if self.use_attention:
                x = self.attention_blocks[i](x)
        
        return x


In [ ]:
class UNet(nn.Module):
    """
    U-Net architecture for DDPM noise prediction.
    Scalable toward Stable Diffusion with attention mechanisms and flexible architecture.
    
    Args:
        in_channels: Number of input channels (3 for RGB images)
        out_channels: Number of output channels (same as input for noise prediction)
        base_channels: Base number of channels (multiplied by channel_mults)
        channel_mults: Tuple of channel multipliers for each resolution level
        num_res_blocks: Number of residual blocks per resolution level
        time_emb_dim: Dimension of time embedding
        use_attention_levels: Which resolution levels to use self-attention (tuple of bools)
        num_heads: Number of attention heads
        dropout: Dropout rate
    """
    def __init__(
        self,
        in_channels=3,
        out_channels=3,
        base_channels=64,
        channel_mults=(1, 2, 2, 4),
        num_res_blocks=2,
        time_emb_dim=256,
        use_attention_levels=(False, False, True, True),
        num_heads=4,
        dropout=0.0
    ):
        super(UNet, self).__init__()
                   
        self.time_emb_dim = time_emb_dim
        self.base_channels = base_channels
        
        # Calculate channel sizes for each level
        self.channels = [base_channels * mult for mult in channel_mults]
             
        # Time embedding module
        self.time_embedding = TimeEmbedding(time_emb_dim)
 
        # Initial convolution
        self.init_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        
        # Encoder (downsampling path)
        self.encoder_blocks = nn.ModuleList()
        in_ch = base_channels
        for i, out_ch in enumerate(self.channels):
            self.encoder_blocks.append(
                EncoderBlock(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    time_emb_dim=time_emb_dim,
                    num_res_blocks=num_res_blocks,
                    downsample=True,
                    use_attention=use_attention_levels[i] if i < len(use_attention_levels) else False,
                    num_heads=num_heads,
                    dropout=dropout
                )
            )
            in_ch = out_ch
        
        # Bottleneck (middle blocks with attention)
        bottleneck_ch = self.channels[-1]
        self.bottleneck = nn.ModuleList([
            ResBlockFiLM(bottleneck_ch, bottleneck_ch, time_emb_dim, dropout=dropout),
            SelfAttention(bottleneck_ch, num_heads),
            ResBlockFiLM(bottleneck_ch, bottleneck_ch, time_emb_dim, dropout=dropout)
        ])
        
        # Decoder (upsampling path)
        self.decoder_blocks = nn.ModuleList()
        reversed_channels = list(reversed(self.channels))
        reversed_attention = list(reversed(use_attention_levels))
        
        for i, out_ch in enumerate(reversed_channels):
            self.decoder_blocks.append(
                DecoderBlock(
                    in_channels=in_ch,
                    out_channels= out_ch,
                    time_emb_dim=time_emb_dim,
                    num_res_blocks=num_res_blocks,
                    upsample=True,
                    use_attention=reversed_attention[i] if i < len(reversed_attention) else False,
                    num_heads=num_heads,
                    dropout=dropout
                )
            )
            in_ch = out_ch
        
        # Final output layers
        self.final_norm = nn.GroupNorm(8, base_channels)
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=3, padding=1)
        
    def forward(self, x, t):
        """
        Forward pass through U-Net.
        
        Args:
            x: Noisy input image (B, in_channels, H, W)
            t: Timestep tensor (B,) with values in range [0, num_timesteps-1]
        Returns:
            Predicted noise (B, out_channels, H, W)
        """
        # Time embedding
        time_emb = self.time_embedding(t)  # (B, time_emb_dim)
        
        # Initial convolution
        x = self.init_conv(x)  # (B, base_channels, H, W)
        
        # Encoder path
        skip_connections = []
        for encoder_block in self.encoder_blocks:
            x, skip = encoder_block(x, time_emb)
            skip_connections.append(skip)
        
        # Bottleneck
        for bottleneck_block in self.bottleneck:
            if isinstance(bottleneck_block, ResBlockFiLM):
                x = bottleneck_block(x, time_emb)
            else:  # SelfAttentionBlock
                x = bottleneck_block(x)
        
        # Decoder path
        for decoder_block in self.decoder_blocks:
            skip = skip_connections.pop()
            x = decoder_block(x, skip, time_emb)
        
        # Final output
        x = self.final_norm(x)
        x = torch.nn.functional.silu(x)
        x = self.final_conv(x)
        
        return x

In [ ]:
class DDPM:
    """  
    Diffusion model for image generation.
    
    """
    def __init__(self, n_steps = 1000, beta_start = 0.0001, beta_end = 0.2, schedule_type = 'linear',device = 'cuda'):
        """
        Args: 
            n_steps : Total number of diffusion steps T
            beta_start : Starting value of beta (noise variance)
            beta_end : Ending value of beta
            schedule_type : "linear" or "cosine"
            device : Device to store the tensors       
        """
        self.n_steps = n_steps
        self.device = device
        
        # initialize beta schedule 
        if schedule_type == 'linear':
            betas = torch.linspace(beta_start,beta_end,n_steps,dtype=torch.float32)
        elif schedule_type == 'cosine':
            betas = self._cosine_beta_schedule(n_steps)
        else:
            raise NotImplementedError(f"Unknown schedule type: {schedule_type}")
        
        # Precompute alpha values for efficient sampling
        self.betas = betas.to(device)
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alpha_cumprod_prev = F.pad(self.alpha_cumprod[:-1], (1, 0), value=1.0)  # ᾱ_{t-1}
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_cumprod = torch.sqrt(1.0 - self.alpha_cumprod)
        
        # Precomputing posterior mean coefficients  and variance for sampling
        self.posterior_mean_coef1 = ( #! coeff of xt 
            torch.sqrt(1.0 / self.alphas)
        )
        self.posterior_mean_coef2 = ( #! coeff of eps_theta
            -self.betas / (torch.sqrt(self.alphas) *torch.sqrt(1.0 - self.alpha_cumprod))
        
        )
        self.posterior_variance = self.betas * (1.0 - self.alpha_cumprod_prev) / (1.0 - self.alpha_cumprod)
        
    def _cosine_beta_schedule(self, timesteps, s=0.008):
        """Cosine schedule as proposed in https://arxiv.org/abs/2102.09672"""
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps, dtype=torch.float32)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clamp(betas, 0.0001, 0.9999)
    
    def q_sample(self, x_start, t, noise=None):
        """
        Diffuse the data (add noise) at timestep t.
        Forward diffusion : 
            q(x_t | x_0) = N(x_t; sqrt(ᾱ_t) * x_0, (1 - ᾱ_t) * I)
        
        Args:
            x_start: Original clean image (B, C, H, W)
            t: Timestep tensor (B,) with values in range [0, n_steps-1]
            noise: Optional noise to add (if None, sampled from standard normal)
        Returns:
            Noisy image at timestep t
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alpha_cumprod = self.sqrt_alpha_cumprod[t][:, None, None, None]
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_cumprod[t][:, None, None, None]
        
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise
    def p_sample(self,model,x_t,t):
        """ 
        Sample from the reverse process (denoise).
        Reverse diffusion:
            p(x_{t-1} | x_t) = N(x_{t-1}; μ_θ(x_t, t), σ² * I)
        where μ_θ is predicted by the model and σ² is derived from the noise schedule.
        Args:
            model: The noise prediction model (e.g., UNet)
            x_t: Noisy image at timestep t (B, C, H, W)
            t: Timestep tensor (B,) with values in range [0, n_steps-1]
        
        """
        # Predict noise using the model
        predicted_noise = model(x_t, t)
        posterior_mean = (self.posterior_mean_coef1[t][:, None, None, None] * x_t 
                          + self.posterior_mean_coef2[t][:, None, None, None] * predicted_noise
        )
        posterior_variance = self.posterior_variance[t][:, None, None, None]
        noise = torch.randn_like(x_t) if t[0] > 0 else torch.zeros_like(x_t)  # No noise at t=0
        x_prev = posterior_mean + torch.sqrt(posterior_variance) * noise
        return x_prev
    @torch.no_grad()
    def sample(self,model,shape,return_snaps = False):
        """ 
        Generate a sample from the model by iteratively applying p_sample from T to 0.
        Args:
            model: The noise prediction model (e.g., UNet)
            shape: Shape of the generated image (B, C, H, W)
            return_snaps: If True, return intermediate samples at certain timesteps for visualization
        Returns:
            Generated image (B, C, H, W) and optionally intermediate snapshots
        """
        device = next(model.parameters()).device
        x_t = torch.randn(shape, device= device)  # Start from pure noise
        snaps = []
        for t in reversed(range(self.n_steps)):
            t_tensor = torch.full((shape[0],), t, device=device, dtype=torch.long)
            x_t = self.p_sample(model, x_t, t_tensor)
            if return_snaps and t % (self.n_steps // 10) == 0:  # Save snapshots at intervals
                snaps.append(x_t.cpu().clone())
        if return_snaps:
            return x_t, snaps
        return (x_t,snaps) if return_snaps else (x_t, None)
        

In [ ]:
# plot the betas and alphas 
import matplotlib.pyplot as plt
def plot_schedules(ddpm):
    timesteps = torch.arange(ddpm.n_steps)
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(timesteps.cpu(), ddpm.betas.cpu(), label='Beta Schedule')
    plt.title('Beta Schedule')
    plt.xlabel('Timestep')
    plt.ylabel('Beta Value')
    plt.grid()
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(timesteps.cpu(), ddpm.alpha_cumprod.cpu(), label='Cumulative Product of Alphas')
    plt.title('Cumulative Product of Alphas (ᾱ_t)')
    plt.xlabel('Timestep')
    plt.ylabel('ᾱ_t Value')
    plt.grid()
    plt.legend()
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    ddpm = DDPM(n_steps=1000, beta_start=0.0001, beta_end=0.02, schedule_type='linear')
    plot_schedules(ddpm)

In [ ]:
class EMA:
    """ 
    Exponential Moving Average (EMA) wrapper for model parameters.
    
         ema = decay * ema + (1 - decay) * current_params
    
    """
    def __init__(self, model, decay=0.999,update_every=1):
        import copy
        self.ema_model = copy.deepcopy(model)
        self.decay = decay
        self.update_every = update_every
        self.ema_model.eval()  # EMA model is used for inference, so set to eval mode
        for p in self.ema_model.parameters():
            p.requires_grad_(False)
        self.step = 0
    def update(self, model):
        """Update EMA parameters with current model parameters."""
        self.step += 1
        if self.step % self.update_every == 0:
            with torch.no_grad():
                # update parameters
                for ema_param, model_param in zip(self.ema_model.parameters(), model.parameters()):
                    ema_param.mul_(self.decay).add_(model_param, alpha=1 - self.decay)
                
                # update the buffers too for sync (e.g., for normalization layers)  
                for ema_buffer, model_buffer in zip(self.ema_model.buffers(), model.buffers()):
                    ema_buffer.copy_(model_buffer)
    def get_model(self):
        return self.ema_model
    def state_dict(self):
        return self.ema_model.state_dict()
    def load_state_dict(self, state_dict):
        self.ema_model.load_state_dict(state_dict)
        
    
    

## Diffusion Process and Training

In [ ]:
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import torchvision.utils as vutils

def generate_samples_at_noise_levels(model, ddpm, real_images, device, num_samples=6):
    """
    Test model's ability to recover original images from different noise levels.
    Takes real images, adds noise at different intensities, then denoises them back.
    
    Args:
        model: UNet model
        ddpm: DDPM diffusion process
        real_images: Real images from dataset (B, C, H, W), already normalized to [-1, 1]
        device: Device to run on
        num_samples: Number of images to test (uses first num_samples from batch)
    
    Returns:
        Figure showing: original, noisy at different t, and recovered images
    """
    model.eval()
    
    # Different noise levels to test (from no noise to high noise)
    noise_levels = [100, 250, 500, 750, 999]  # timesteps
    
    # Each row: original, noisy@t, recovered for each noise level
    fig, axes = plt.subplots(len(noise_levels), num_samples * 3, 
                            figsize=(num_samples * 3 * 1.5, len(noise_levels) * 1.5))
    
    real_images = real_images[:num_samples].to(device)
    
    with torch.no_grad():
        for level_idx, t_noise in enumerate(noise_levels):
            # Add noise at timestep t_noise (forward diffusion)
            t_tensor = torch.full((num_samples,), t_noise, device=device, dtype=torch.long)
            noise = torch.randn_like(real_images)
            noisy_images = ddpm.q_sample(real_images, t_tensor, noise)
            
            # Denoise back to t=0 (reverse diffusion)
            x_t = noisy_images.clone()
            for t in reversed(range(t_noise + 1)):
                t_tensor = torch.full((num_samples,), t, device=device, dtype=torch.long)
                x_t = ddpm.p_sample(model, x_t, t_tensor)
            
            recovered_images = x_t
            
            # Visualize: original, noisy, recovered for each sample
            for i in range(num_samples):
                # Original image
                orig_img = (real_images[i].cpu() + 1) / 2
                orig_img = torch.clamp(orig_img, 0, 1)
                ax_orig = axes[level_idx, i * 3]
                ax_orig.imshow(orig_img.permute(1, 2, 0).numpy())
                ax_orig.axis('off')
                if level_idx == 0:
                    ax_orig.set_title('Original', fontsize=9, fontweight='bold')
                if i == 0:
                    ax_orig.text(-0.1, 0.5, f't={t_noise}', transform=ax_orig.transAxes,
                               fontsize=10, fontweight='bold', va='center', rotation=90)
                
                # Noisy image
                noisy_img = (noisy_images[i].cpu() + 1) / 2
                noisy_img = torch.clamp(noisy_img, 0, 1)
                ax_noisy = axes[level_idx, i * 3 + 1]
                ax_noisy.imshow(noisy_img.permute(1, 2, 0).numpy())
                ax_noisy.axis('off')
                if level_idx == 0:
                    ax_noisy.set_title('Noisy', fontsize=9, fontweight='bold')
                
                # Recovered image
                recov_img = (recovered_images[i].cpu() + 1) / 2
                recov_img = torch.clamp(recov_img, 0, 1)
                ax_recov = axes[level_idx, i * 3 + 2]
                ax_recov.imshow(recov_img.permute(1, 2, 0).numpy())
                ax_recov.axis('off')
                if level_idx == 0:
                    ax_recov.set_title('Recovered', fontsize=9, fontweight='bold')
    
    plt.suptitle('Model Recovery from Different Noise Levels', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    model.train()
    return fig


def generate_full_samples(model, ddpm, device, num_samples=16, image_size=128):
    """
    Generate completely new samples from pure noise (full denoising process).
    
    Args:
        model: UNet model
        ddpm: DDPM diffusion process
        device: Device to run on
        num_samples: Number of samples to generate
        image_size: Size of generated images
    
    Returns:
        Figure showing generated samples
    """
    model.eval()
    
    with torch.no_grad():
        # Generate samples from pure noise
        samples, _ = ddpm.sample(model, (num_samples, 3, image_size, image_size), return_snaps=False)
        
        # Denormalize from [-1, 1] to [0, 1]
        samples = (samples + 1) / 2
        samples = torch.clamp(samples, 0, 1)
    
    # Create a grid of images
    grid = vutils.make_grid(samples.cpu(), nrow=4, padding=2, normalize=False)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis('off')
    ax.set_title('Generated Samples (Full Denoising from Pure Noise)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    model.train()
    return fig


In [ ]:
def train(model, train_dataloader, ddpm, optimizer, device, num_epochs=10, ema = None,
          checkpoint_dir='./checkpoints', samples_dir='./samples', save_every=1, 
          generate_samples_every=10, image_size=128, amp=True, accum_steps=1, grad_clip=None,
          checkpoint_interval_minutes=30, use_ema_for_sampling = False):
    """
    Train the DDPM model with checkpointing and sample generation for monitoring progress.
    
    Args:
        model: UNet model for noise prediction
        train_dataloader: Training DataLoader
        ddpm: DDPM diffusion process
        optimizer: Optimizer (e.g., Adam)
        device: Device to run training on (cuda/cpu)
        num_epochs: Total number of training epochs
        checkpoint_dir: Directory to save model checkpoints
        samples_dir: Directory to save generated sample images
        save_every: Save checkpoint every N epochs (default: 1)
        generate_samples_every: Generate samples every N epochs (default: 10)
        image_size: Size of images for sample generation
        amp: Enable mixed precision training on CUDA
        accum_steps: Gradient accumulation steps
        grad_clip: Optional max grad norm for clipping
        checkpoint_interval_minutes: Save intermediate checkpoint every N minutes (default: 30)
    
    Returns:
        Dictionary containing training history (losses)
    """
    import time
    from collections import deque
    
    # Create directories if they don't exist
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
        print(f"✅ Created checkpoint directory: {checkpoint_dir}")
    else:
        print(f"✅ Using existing checkpoint directory: {checkpoint_dir}")
    
    if not os.path.exists(samples_dir):
        os.makedirs(samples_dir)
        print(f"✅ Created samples directory: {samples_dir}")
    else:
        print(f"✅ Using existing samples directory: {samples_dir}")
    
    # Get a fixed batch of real images for consistent validation
    validation_images = None
    for batch in train_dataloader:
        validation_images = batch[:6]  # Take 6 images for validation
        break
    
    # Training history
    history = {
        'train_loss': [],
        'batch_losses': [],
        'grad_norms': []
    }
    
    model.train()
    use_amp = amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device='cuda', enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)
    
    # Use deque for efficient window of recent values
    recent_losses = deque(maxlen=500)
    recent_grad_norms = deque(maxlen=500)
    global_step = 0
    
    # Dynamic plotting - create figure and line objects ONCE, then update data only
    plt.ion()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Training Metrics (Live)', fontsize=14, fontweight='bold')
    
    # Create line objects that will be reused (not recreated)
    line1, = ax1.plot([], [], 'b-', alpha=0.7, linewidth=1)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss (MSE)')
    ax1.set_title('Batch Loss')
    ax1.grid(True, alpha=0.3)
    
    line2, = ax2.plot([], [], 'r-', alpha=0.7, linewidth=1)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Gradient Norm')
    ax2.set_title('Gradient Norm')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    def update_plots():
        """Update plots dynamically by modifying line data (efficient - no recreation)"""
        try:
            if len(recent_losses) > 0:
                steps = list(range(global_step - len(recent_losses) + 1, global_step + 1))
                line1.set_data(steps, list(recent_losses))
                ax1.relim()
                ax1.autoscale_view()
            
            if len(recent_grad_norms) > 0:
                steps = list(range(global_step - len(recent_grad_norms) + 1, global_step + 1))
                line2.set_data(steps, list(recent_grad_norms))
                ax2.relim()
                ax2.autoscale_view()
            
            fig.canvas.draw()
            fig.canvas.flush_events()
        except Exception:
            pass
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        step_in_epoch = 0
        
        # Time tracking for intermediate checkpoints
        last_checkpoint_time = time.time()
        epoch_start_time = time.time()
        
        # Training loop with progress bar
        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", dynamic_ncols=True)
        for batch in pbar:
            batch = batch.to(device, non_blocking=True)
            B = batch.size(0)
            t = torch.randint(0, ddpm.n_steps, (B,), device=device)
            noise = torch.randn_like(batch)
            x_t = ddpm.q_sample(batch, t, noise)
            
            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                predicted_noise = model(x_t, t)
                raw_loss = F.mse_loss(predicted_noise, noise)
                loss = raw_loss / accum_steps
            
            scaler.scale(loss).backward()
            step_in_epoch += 1
            
            # Gradient step with norm tracking
            grad_norm = 0.0
            if step_in_epoch % accum_steps == 0:
                if grad_clip is not None:
                    scaler.unscale_(optimizer)
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip).item()
                else:
                    # Calculate grad norm even without clipping
                    scaler.unscale_(optimizer)
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),torch.inf).item()
                
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
                # update ema
                ema.update(model)
                
                
                # Track gradient norm
                recent_grad_norms.append(grad_norm)
                history['grad_norms'].append(grad_norm)
            
            epoch_loss += raw_loss.item()
            num_batches += 1
            global_step += 1
            
            # Track loss
            recent_losses.append(raw_loss.item())
            history['batch_losses'].append(raw_loss.item())
            
            # Update progress bar
            postfix_dict = {
                "Loss": f"{raw_loss.item():.4f}", 
                "Avg": f"{epoch_loss / num_batches:.4f}",
                "EMA steps": ema.step
            }
            if grad_norm > 0:
                postfix_dict["GradNorm"] = f"{grad_norm:.3f}"
            pbar.set_postfix(postfix_dict)
            
            # Update plots every 10 batches
            if global_step % 10 == 0:
                update_plots()
            
            # Save intermediate checkpoint every checkpoint_interval_minutes
            current_time = time.time()
            elapsed_minutes = (current_time - last_checkpoint_time) / 60.0
            if elapsed_minutes >= checkpoint_interval_minutes:
                epoch_checkpoint_path = os.path.join(checkpoint_dir, f'model_epoch_{epoch+1:04d}.pt')
                torch.save({
                    'epoch': epoch + 1,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'ema_state_dict': ema.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': epoch_loss / num_batches,
                    'history': history
                }, epoch_checkpoint_path)
                elapsed_time = (current_time - epoch_start_time) / 60.0
                print(f"\n💾 [Time: {elapsed_time:.1f}min] Saved checkpoint: {epoch_checkpoint_path}")
                last_checkpoint_time = current_time
        
        # Flush remaining gradients if accumulation steps do not divide batches
        if accum_steps > 1 and step_in_epoch % accum_steps != 0:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            
            ema.update(model)
        
        # Calculate average training loss for this epoch
        avg_train_loss = epoch_loss / num_batches
        history['train_loss'].append(avg_train_loss)
        epoch_duration = (time.time() - epoch_start_time) / 60.0
        print(f"\nEpoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Duration: {epoch_duration:.1f}min")
        
        # Save final epoch checkpoint every save_every epochs
        if (epoch + 1) % save_every == 0:
            epoch_checkpoint_path = os.path.join(checkpoint_dir, f'model_epoch_{epoch+1:04d}.pt')
            torch.save({
                'epoch': epoch + 1,
                'global_step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'ema_state_dict': ema.state_dict(),
                'train_loss': avg_train_loss,
                'history': history
            }, epoch_checkpoint_path)
            print(f"💾 Saved final epoch checkpoint: {epoch_checkpoint_path}")
        
        # Generate samples every generate_samples_every epochs to monitor progress
        if (epoch + 1) % generate_samples_every == 0:
            sampler = ema.get_model() if use_ema_for_sampling else model
            print(f"🎨 Generating samples at epoch {epoch+1}...")
            
            # Close training plot temporarily
            if fig is not None:
                plt.ioff()
                plt.close(fig)
            
            # Test recovery from different noise levels using real images
            if validation_images is not None:
                
                fig_recovery = generate_samples_at_noise_levels(sampler, ddpm, validation_images, device, num_samples=6)
                recovery_path = os.path.join(samples_dir, f'recovery_epoch_{epoch+1:04d}.png')
                fig_recovery.savefig(recovery_path, dpi=150, bbox_inches='tight')
                plt.close(fig_recovery)
                print(f"  💾 Saved recovery test: {recovery_path}")
            
            # Generate full samples from pure noise
            fig_samples = generate_full_samples(sampler, ddpm, device, num_samples=16, image_size=image_size)
            samples_path = os.path.join(samples_dir, f'samples_epoch_{epoch+1:04d}.png')
            fig_samples.savefig(samples_path, dpi=150, bbox_inches='tight')
            plt.close(fig_samples)
            print(f"  💾 Saved generated samples: {samples_path}")
            
            # Re-enable interactive plotting
            plt.ion()
    
    # Close all plots
    plt.ioff()
    if fig is not None:
        plt.close(fig)
    
    # Save final model
    final_checkpoint_path = os.path.join(checkpoint_dir, 'model_final.pt')
    torch.save({
        'epoch': num_epochs,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history
    }, final_checkpoint_path)
    print(f"✅ Training complete! Final model saved: {final_checkpoint_path}")
    
    sampler = ema.get_model() if use_ema_for_sampling else model
    # Generate final samples
    print(f"🎨 Generating final samples...")
    if validation_images is not None:
        fig_recovery = generate_samples_at_noise_levels(sampler, ddpm, validation_images, device, num_samples=6)
        fig_recovery.savefig(os.path.join(samples_dir, 'recovery_final.png'), dpi=150, bbox_inches='tight')
        plt.close(fig_recovery)
    
    fig_samples = generate_full_samples(model, ddpm, device, num_samples=16, image_size=image_size)
    fig_samples.savefig(os.path.join(samples_dir, 'samples_final.png'), dpi=150, bbox_inches='tight')
    plt.close(fig_samples)
    print(f"💾 Saved final samples to {samples_dir}")
    
    return history

In [ ]:
def load_checkpoint(model, checkpoint_path, optimizer=None, device='cuda'):
    """
    Load model checkpoint.
    
    Args:
        model: UNet model to load weights into
        checkpoint_path: Path to checkpoint file
        optimizer: Optional optimizer to load state
        device: Device to load model to
    
    Returns:
        Tuple of (model, optimizer, epoch, history)
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded model from epoch {checkpoint['epoch']}")
    
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"✅ Loaded optimizer state")
    
    epoch = checkpoint.get('epoch', 0)
    history = checkpoint.get('history', {'train_loss': []})
    
    return model, optimizer, epoch, history



In [ ]:

def plot_training_history(history):
    """
    Plot training loss curve.
    
    Args:
        history: Dictionary containing 'train_loss' list
    """
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss (MSE)', fontsize=12)
    ax.set_title('DDPM Training History', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# create dataloader 

batch_size = 24
train_dataloader = DataLoader(dataset,batch_size = batch_size,shuffle = True, num_workers = 4, pin_memory = True)
print(f"Train dataset size: {len(train_dataloader)}")

model = UNet(
    in_channels = 3,
    out_channels = 3,
    base_channels = 128,
    channel_mults = (1,2,2,4),
    num_res_blocks = 2,
    time_emb_dim = 256,
    use_attention_levels=(False, True, True, True),
    num_heads=4,
    dropout = 0.1,
    
).to(device)

ddpm = DDPM(
    n_steps = 1000,
    beta_start = 0.0001,
    beta_end = 0.02,
    schedule_type = "cosine",
    device = device
)



In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

In [ ]:
optimizer.step

In [ ]:
# Initialize EMA for stable training and better sample quality
ema = EMA(model, decay=0.99, update_every=1)
print(f"✅ Initialized EMA with decay={ema.decay}")

In [ ]:
# model, optimizer, _, _ = load_checkpoint(model,r'./checkpoints/model_epoch_0001.pt',optimizer , device=device)

In [ ]:
optimizer.step

In [ ]:
history = train(
    model=model,
    train_dataloader=train_dataloader,
    ddpm=ddpm,
    optimizer=optimizer,
    device=device,
    num_epochs=10,
    ema = ema,
    checkpoint_dir='./checkpoints',
    samples_dir='./samples',
    save_every=1,  # Save checkpoint every epoch
    generate_samples_every=1,  # Generate samples every epoch
    image_size=128,
    amp=True,
    accum_steps=4,
    grad_clip=1.0,
    checkpoint_interval_minutes=30,  # Save intermediate checkpoint every 30 minutes (overwritten)
    use_ema_for_sampling = True
)


In [ ]:
epoch = 1
checkpoint_dir='/kaggle/working/'
checkpoint_path = os.path.join(checkpoint_dir, f'model_epoch_{epoch+1:04d}.pt')
torch.save({
    'epoch': epoch + 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, checkpoint_path)
print(f"💾 Saved checkpoint: {checkpoint_path}")

In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/model_epoch_0005.pt')

In [ ]:
from IPython.display import FileLink
FileLink('./checkpoints/model_epoch_0005.pt')

In [ ]:
plot_training_history(history)